In [26]:
import time
import os
import copy
import urllib.request
import shutil
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, transforms
from tensorflow.keras import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Concatenate, Dense, Flatten, Dropout, Layer, Lambda
from tensorflow.image import resize_with_pad
import tensorflow as tf
import logging
import warnings
tf.get_logger().setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

#Data
if not os.path.exists('inspiritai_util.py'):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/inspirit-ai-data-bucket-1/Modules/inspiritai_util.py",
        "inspiritai_util.py",
    )

download_url = "https://storage.googleapis.com/inspirit-ai-data-bucket-1/Data/AI%20Scholars/Sessions%206%20-%2010%20(Projects)/Project%20-%20Towards%20Precision%20Medicine/"

data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

for filename in ["images.npy", "labels.npy"]:
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        urllib.request.urlretrieve(download_url + filename, file_path)

images = np.load(os.path.join(data_dir, "images.npy"))
labels = np.load(os.path.join(data_dir, "labels.npy"))


# Pennylane
import pennylane as qml
from pennylane import numpy as pnp

# Plotting
import matplotlib.pyplot as plt

In [27]:
labels_ohe = np.array(pd.get_dummies(labels))
y = labels_ohe
X = images / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=labels
)
print(X_train.shape)


(768, 150, 150, 3)


In [28]:
def ResizeImages(images, height, width):
  return np.array([resize_with_pad(image, height, width, antialias=True) for image in images]).astype(np.float32)

X_train_unnormed = X_train * 255.0
X_test_unnormed = X_test * 255.0

X_train_resized = ResizeImages(X_train_unnormed, 224, 224)
X_test_resized = ResizeImages(X_test_unnormed, 224, 224)

X_train_resized = tf.keras.applications.resnet50.preprocess_input(X_train_resized)
X_test_resized = tf.keras.applications.resnet50.preprocess_input(X_test_resized)


In [35]:
import random
from numpy import flipud, fliplr
from skimage.transform import rotate
from skimage.exposure import adjust_gamma, rescale_intensity
def create_random_augmented_image(original_image):
    choice = random.randint(0, 3)
    if choice == 0:
        new_image = flipud(original_image)
    elif choice == 1:
        new_image = fliplr(original_image)
    elif choice == 2:
        new_image = rotate(original_image, angle=180, resize=False)
    else:
        new_image = rescale_intensity(original_image)

    return new_image

In [36]:
X_train_augment_list = []
y_train_augment_list = []

for i in range(len(X_train_resized)):

    original_image = X_train_resized[i]
    original_label = y_train[i]

    new_augmented_image = create_random_augmented_image(original_image)
    new_augmented_label = original_label

    X_train_augment_list.append(new_augmented_image)
    y_train_augment_list.append(new_augmented_label)

X_train_augment = np.array(X_train_augment_list)
y_train_augment = np.array(y_train_augment_list)

X_train_combined = np.concatenate((X_train_resized, X_train_augment), axis=0)
y_train_combined = np.concatenate((y_train, y_train_augment), axis=0)

In [37]:
class QuantumLayer(Layer):
    def __init__(self, n_qubits, n_layers, **kwargs):
        super().__init__(**kwargs)
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(self.dev, interface="tf", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
            return tuple(qml.expval(qml.PauliZ(i)) for i in range(n_qubits))

        self.circuit = circuit

    def build(self, input_shape):
        self.q_weights = self.add_weight(
            name="q_weights",
            shape=(self.n_layers, self.n_qubits),
            initializer=tf.keras.initializers.RandomUniform(0.0, np.pi),
            trainable=True,
            dtype=tf.float64,
        )

    def _run_one(self, x, q_weights):
        result = self.circuit(x, q_weights)
        result = tf.stack(result)
        result = tf.math.real(result)
        return tf.cast(result, tf.float32)

    def call(self, inputs):
        inputs = tf.cast(inputs, tf.float64)
        q_weights = tf.cast(tf.convert_to_tensor(self.q_weights), tf.float64)

        if hasattr(inputs, "numpy"):
            outputs = [self._run_one(x, q_weights) for x in tf.unstack(inputs)]
            outputs = tf.stack(outputs)
        else:
            outputs = tf.map_fn(
                lambda x: self._run_one(x, q_weights),
                inputs,
                fn_output_signature=tf.TensorSpec(shape=(self.n_qubits,), dtype=tf.float32),
            )

        outputs.set_shape((None, self.n_qubits))
        return outputs

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.n_qubits)


In [38]:
n_qubits = 8
n_layers = 1

feature_extractor = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
    pooling='avg',
)

feature_extractor.trainable = False

X_train_features = feature_extractor.predict(X_train_combined, batch_size=16, verbose=1)
X_test_features = feature_extractor.predict(X_test, batch_size=16, verbose=1)

feature_mean = X_train_features.mean(axis=0, keepdims=True)
feature_std = X_train_features.std(axis=0, keepdims=True) + 1e-6
X_train_features = (X_train_features - feature_mean) / feature_std
X_test_features = (X_test_features - feature_mean) / feature_std

feature_input = tf.keras.Input(shape=(2048,), name='resnet_features')

x = Dense(64, activation='relu', name='pre_quantum_dense')(feature_input)
x = Dropout(0.2, name='pre_quantum_dropout')(x)
bottleneck = Dense(n_qubits, activation='tanh', name='quantum_bottleneck')(x)
angles = Lambda(lambda z: z * np.pi, name='angle_scaling')(bottleneck)
quantum_outputs = QuantumLayer(n_qubits, n_layers, name='quantum_layer')(angles)

x = Concatenate(name='quantum_residual')([bottleneck, quantum_outputs])
x = Dense(32, activation='relu', name='post_quantum_dense')(x)
qnn_output = Dense(8, activation='softmax', name='tissue_output')(x)

transfer_qnn = Model(feature_input, qnn_output)

transfer_qnn.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy', 'categorical_crossentropy'],
    run_eagerly=True,
)

transfer_qnn.summary()


96/96 ━━━━━━━━━━━━━━━━━━━━ 27s 270ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 133ms/step


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ resnet_features     │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pre_quantum_dense   │ (None, 64)        │    131,136 │ resnet_features[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pre_quantum_dropout │ (None, 64)        │          0 │ pre_quantum_dens… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantum_bottleneck  │ (None, 8)         │        520 │ pre_quantum_drop… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ angle_scaling       │ (None, 8)         │          0 │ quantum_bottlene… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantum_layer       │ (None, 8)         │          8 │ angle_scaling[0]… │
│ (QuantumLayer)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantum_residual    │ (None, 16)        │          0 │ quantum_bottlene… │
│ (Concatenate)       │                   │            │ quantum_layer[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ post_quantum_dense  │ (None, 32)        │        544 │ quantum_residual… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tissue_output       │ (None, 8)         │        264 │ post_quantum_den… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 132,472 (517.50 KB)

 Trainable params: 132,472 (517.50 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
history = transfer_qnn.fit(
    X_train_features,
    y_train_combined,
    epochs=15,
    batch_size=16,
    validation_data=(X_test_features, y_test),
)

Epoch 1/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 148s 2s/step - accuracy: 0.6660 - categorical_crossentropy: 1.2012 - loss: 1.2012 - val_accuracy: 0.1250 - val_categorical_crossentropy: 2.3413 - val_loss: 2.3413
Epoch 2/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 150s 2s/step - accuracy: 0.7669 - categorical_crossentropy: 0.8614 - loss: 0.8614 - val_accuracy: 0.1211 - val_categorical_crossentropy: 2.4974 - val_loss: 2.4974
Epoch 3/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 148s 2s/step - accuracy: 0.8216 - categorical_crossentropy: 0.6611 - loss: 0.6611 - val_accuracy: 0.1133 - val_categorical_crossentropy: 2.3796 - val_loss: 2.3796
Epoch 4/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - accuracy: 0.8385 - categorical_crossentropy: 0.5424 - loss: 0.5424 - val_accuracy: 0.1133 - val_categorical_crossentropy: 2.1970 - val_loss: 2.1970
Epoch 5/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 151s 2s/step - accuracy: 0.8548 - categorical_crossentropy: 0.4925 - loss: 0.4925 - val_accuracy: 0.1406 - val_categorical_crossentropy: 2.2272 - val_loss: 2.

KeyboardInterrupt: 